# Swahili News Topic Classification: Data + Baseline

Starter notebook for the African-language text classification assignment.

**What this notebook covers:** loading the data, inspecting it, cleaning, splitting into train/validation/test, a TF-IDF + logistic regression baseline, metrics, error analysis, and saving the model for the web app.

**Next notebooks:** BiGRU/BiLSTM with fastText vectors, then AfroXLMR / AfriBERTa fine-tuning. Reuse the same splits so the comparison is fair.

## 1. Setup

In [1]:
%pip install -q datasets scikit-learn pandas matplotlib seaborn joblib


You should consider upgrading via the '/Users/macbookpro/Documents/ALU/alu-machine_learning/swahili_txt_classification/.venv/bin/python -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import re, random, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, f1_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Paths are relative to the repo root, whether this runs from notebooks/ or from Colab's /content.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODELS_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results" / "baseline"
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

/Users/macbookpro/Documents/ALU/alu-machine_learning/swahili_txt_classification/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/macbookpro/Documents/ALU/alu-machine_learning/swahili_txt_classification/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Load the data

The Swahili news dataset has a `train` and a `test` split. We keep the official test set untouched and carve a validation set out of train.

If the first load fails (dataset hosting changes over time), the fallback tries the original name. Check the dataset card on Hugging Face for the exact class counts and cite them in your report.

In [3]:
try:
    ds = load_dataset("community-datasets/swahili_news")
except Exception as e:
    print("First attempt failed:", e)
    ds = load_dataset("swahili_news")

print(ds)
label_names = ds["train"].features["label"].names
print("Classes:", label_names)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 22207
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7338
    })
})
Classes: ['uchumi', 'kitaifa', 'michezo', 'kimataifa', 'burudani', 'afya']


In [4]:
train_full = ds["train"].to_pandas()
test_df    = ds["test"].to_pandas()

# Map integer labels to readable names
train_full["label_name"] = train_full["label"].map(lambda i: label_names[i])
test_df["label_name"]    = test_df["label"].map(lambda i: label_names[i])

train_full.head()

,text,label,label_name
0,"Bodi ya Utalii Tanzania (TTB) imesema, itafan...",0,uchumi
1,"PENDO FUNDISHA-MBEYA RAIS Dk. John Magufuri, ...",1,kitaifa
2,Mwandishi Wetu -Singida BENKI ya NMB imetoa ms...,0,uchumi
3,"TIMU ya taifa ya Tanzania, Serengeti Boys jan...",2,michezo
4,Na AGATHA CHARLES – DAR ES SALAAM ALIYEKUWA K...,1,kitaifa


## 3. Look at the data before touching it

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train_full["label_name"].value_counts().plot.bar(ax=axes[0], color="#4C72B0")
axes[0].set_title("Class distribution (train)")
axes[0].set_ylabel("Articles")

train_full["text"].str.split().str.len().clip(upper=1500).plot.hist(bins=50, ax=axes[1], color="#55A868")
axes[1].set_title("Article length in words (clipped at 1500)")

plt.tight_layout(); plt.show()

print("Duplicate texts in train:", train_full["text"].duplicated().sum())
print("Empty texts in train:", (train_full["text"].str.strip() == "").sum())

Duplicate texts in train: 0
Empty texts in train: 1


/var/folders/lm/_gjv0dgn4pvf8dd2x7pm8yv40000gn/T/ipykernel_79017/3658898659.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. Preprocessing

Keep it light. Swahili is agglutinative (lots of meaning packed into prefixes and suffixes), and aggressive stemming or stopword removal without a proper Swahili tool can hurt more than it helps. What we do:

- lowercase
- remove URLs and email addresses
- remove digits-only tokens and stray symbols
- collapse extra whitespace

Document every step in your report, including the ones you chose *not* to do and why.

In [6]:
URL_RE   = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+")
NONWORD  = re.compile(r"[^a-zA-Z\u00C0-\u017F'\s]")   # keep letters and apostrophes
SPACES   = re.compile(r"\s+")

def clean_text(t: str) -> str:
    t = t.lower()
    t = URL_RE.sub(" ", t)
    t = EMAIL_RE.sub(" ", t)
    t = NONWORD.sub(" ", t)
    t = SPACES.sub(" ", t).strip()
    return t

for df in (train_full, test_df):
    df["clean"] = df["text"].apply(clean_text)

# Drop duplicates and empties from train only (never alter the official test set)
before = len(train_full)
train_full = train_full[train_full["clean"].str.len() > 0].drop_duplicates(subset="clean").reset_index(drop=True)
print(f"Train: {before} -> {len(train_full)} after removing empties/duplicates")

# Check for leakage between train and test
leak = set(train_full["clean"]) & set(test_df["clean"])
print("Texts appearing in both train and test:", len(leak))

Train: 22207 -> 22185 after removing empties/duplicates
Texts appearing in both train and test: 46


## 5. Splits

- **Test:** the official test split, used once at the very end.
- **Train / validation:** stratified 90/10 split of the official train set.

Think of it as practice questions (train), a mock exam (validation, for tuning), and the real exam (test, touched once).

In [7]:
train_df, val_df = train_test_split(
    train_full, test_size=0.10, stratify=train_full["label"], random_state=SEED
)
train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")
pd.DataFrame({
    "train": train_df["label_name"].value_counts(normalize=True),
    "val":   val_df["label_name"].value_counts(normalize=True),
    "test":  test_df["label_name"].value_counts(normalize=True),
}).round(3)

Train: 19966 | Validation: 2219 | Test: 7338


,train,val,test
label_name,,,
kitaifa,0.457,0.457,0.456
michezo,0.269,0.269,0.272
burudani,0.098,0.098,0.096
kimataifa,0.084,0.084,0.082
uchumi,0.057,0.057,0.064
afya,0.035,0.035,0.030


## 6. Baseline: TF-IDF + Logistic Regression

This is your tape measure. Every fancier model has to beat these numbers to justify its cost.

In [8]:
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200_000, sublinear_tf=True)),
    ("clf",   LogisticRegression(max_iter=1000, C=5.0, class_weight="balanced", n_jobs=-1)),
])

import time
t0 = time.perf_counter()
baseline.fit(train_df["clean"], train_df["label"])
train_seconds = time.perf_counter() - t0
print(f"Trained in {train_seconds / 60:.1f} min.")


Trained in 4.0 min.


## 7. Evaluation

In [9]:
def evaluate(model, df, name):
    preds = model.predict(df["clean"])
    acc = accuracy_score(df["label"], preds)
    p, r, f1, _ = precision_recall_fscore_support(df["label"], preds, average="macro", zero_division=0)
    print(f"[{name}] accuracy={acc:.4f}  macro-P={p:.4f}  macro-R={r:.4f}  macro-F1={f1:.4f}")
    return preds

val_preds = evaluate(baseline, val_df, "validation")

[validation] accuracy=0.8891  macro-P=0.8015  macro-R=0.8611  macro-F1=0.8272


In [10]:
# Run on test ONCE, after you are done tuning on validation
test_preds = evaluate(baseline, test_df, "test")
print()
print(classification_report(test_df["label"], test_preds, target_names=label_names, digits=3))

[test] accuracy=0.8803  macro-P=0.7876  macro-R=0.8426  macro-F1=0.8114

              precision    recall  f1-score   support

      uchumi      0.602     0.707     0.650       467
     kitaifa      0.914     0.857     0.885      3348
     michezo      0.967     0.949     0.958      1996
   kimataifa      0.852     0.932     0.890       605
    burudani      0.895     0.930     0.912       703
        afya      0.495     0.680     0.573       219

    accuracy                          0.880      7338
   macro avg      0.788     0.843     0.811      7338
weighted avg      0.889     0.880     0.883      7338



In [11]:
cm = confusion_matrix(test_df["label"], test_preds)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(7, 6))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Baseline confusion matrix (row-normalised)")
plt.tight_layout(); plt.show()

/var/folders/lm/_gjv0dgn4pvf8dd2x7pm8yv40000gn/T/ipykernel_79017/375761430.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 8. Error analysis

Pull the misclassified test articles, sorted by how confident the model was when it got them wrong. Those are the most interesting ones. Read 30 to 50 of them and tag each with a cause, for example:

- article spans two topics (a minister at a football match)
- English or other-language words mixed in
- very short or low-content text
- label looks questionable
- topic words rare in training data

Put your tags in the `cause` column and count them for the report.

In [12]:
proba = baseline.predict_proba(test_df["clean"])
errors = test_df.copy()
errors["pred"]       = [label_names[i] for i in test_preds]
errors["confidence"] = proba.max(axis=1)
errors = errors[errors["label"] != test_preds].sort_values("confidence", ascending=False)
errors["cause"] = ""   # fill in by hand after reading

print(f"{len(errors)} misclassified out of {len(test_df)} ({len(errors)/len(test_df):.1%})")

pd.set_option("display.max_colwidth", 300)
errors[["label_name", "pred", "confidence", "text"]].head(15)

878 misclassified out of 7338 (12.0%)


,label_name,pred,confidence,text
2012,kitaifa,kimataifa,0.996845,"KIONGOZI wa Korea Kaskazini, Kim Jong-un amewaandalia mkutano na chakula cha jioni wajumbe wawili kutoka Korea Kusini, ikiwa ni mara ya kwanza kiongozi huyo kukutana na ujumbe kama huo tangu aingie madarakani mwaka 2011. Shirika la Habari la Korea Kusini la Yonhap, liliripoti kuhusu mkutano huo ..."
443,burudani,kimataifa,0.996322,"Ankara, uturuki UTURUKI imeanza kuwarejesha wapiganaji wa kigeni\nwa kundi linalojiita Dola la Kiislamu (IS) kwenye mataifa yao, yakiwemo\nUjerumani, Denmark na Marekani katika mpango wa kuwarejesha makwao wafungwa wa\nkundi hilo. Ilisema\nimewakamata wanamgambo 287 katika eneo la kaskazini mwa..."
3080,burudani,michezo,0.995546,"BADI MCHOMOLO NA MITANDAO ZIMEBAKI siku 12 kuweza kusherehekea mwaka mpya wa 2017 na kuuaga mwaka 2016, ndani ya siku 365 za mwaka 2016 kuna mambo mbalimbali yalijitokeza, yakiwa ya furaha na mengine yenye kutia simanzi. Leo hii Mtanzani SPOTIKIKI limekuandalia matukia mbalimbali ambayo yalitok..."
2953,kitaifa,michezo,0.995407,"MECHI za raundi ya nne za michuano ya Ligi ya Mabingwa wa Ulaya zimefanyika wiki hii na kutoa mwelekeo wa hatua ya kwanza ya mtoano ambayo itahusisha jumla ya timu 16.Kwa sasa jumla ya timu 32 zinachuana katika makundi manane, huku kila kundi likiwa na timu nne, ambapo mbili za juu kutoka kila ..."
320,kitaifa,michezo,0.994458,"MSHAMBULIAJI wa Klabu ya Mbeya City FC, Eliud Ambokile amesaini mkataba wa miaka mitatu na nusu kuitumikia timu ya TP Mazembe ya Jamhuri ya Kidemokrasia ya Congo kuanzia msimu ujao.Mazembe ambao ni miongoni mwa timu vigogo Afrika wamenyaka saini ya mchezaji huyo kwa kitita cha Sh milioni 50, ik..."
1587,kitaifa,michezo,0.994328,"LICHA ya kuwafunga waajiri wake wazamani Yanga, Mshambuliaji wa Azam FC , Obrey Chirwa (pichani) ameonesha kutoridhika na hali hiyo na kudai kuwa anaitaka Yanga yenyewe iliyotimia akimaanisha ile ya wakubwa.Chirwa alifunga bao la tatu katika mchezo wa Kombe la Mapinduzi juzi dhidi ya Yanga, mch..."
341,kitaifa,michezo,0.992774,"STAND United imepania ‘kufa’ n a Y anga katika mechi ya Ligi Kuu itakayochezwa kwenye uwanja wa Taifa Dar es Salaam, Jumapili ya wiki hii. Ligi hiyo inaendelea tena baada ya kusimama kwa takribani wiki mbili kuipisha timu ya taifa, Taifa Stars kucheza mechi ya kimataifa dhidi ya Uganda. Kocha m..."
4155,kitaifa,afya,0.992441,"NA VERONICA ROMWALD– DAR ES SALAAM NI wazi kwamba sasa kila mmoja wetu anapaswa kuwa makini na kuzingatia jinsi anavyokula chakula tangu asubuhi hadi anapokwenda kulala usiku. Ingawa wataalamu wa lishe wanaeleza kiuhalisia hakuna chakula, ambacho binadamu anazuiwa kula katika maisha yake hapa d..."
6711,kitaifa,kimataifa,0.991485,"MJUMBE wa Marekani katika Afrika, Tibor Nagy anakwenda Sudan ambako kuna hali tete kutokana na mgogoro wa kisiasa unaoendelea.Nagy ambaye ni Naibu Waziri wa Mambo ya Nje wa Afrika, anakwenda kuomba kusitishwa mashambulio dhidi ya raia. Kutokana na mgogoro huo, kumekuwa na mgomo wa kitaifa wa wa..."
2214,kitaifa,michezo,0.990931,"KWA mara ya mwisho Tanzania ilishiriki fainali za Mataifa ya Afrika kwa wachezaji wa ligi za nyumbani (Chan) mwaka 2009, ikiwa ni miaka 10 sasa tangu wakati huo.Fainali hizo za Chan zilikuwa za kwanza kufanyika tangu kuanzishwa na Shirikisho la Soka la Afrika (Caf) zikiwa na lengo la kuwawezesh..."


In [13]:
# Which class pairs get confused most?
pairs = errors.groupby(["label_name", "pred"]).size().sort_values(ascending=False).head(10)
pairs

label_name  pred     
kitaifa     uchumi       202
            afya         133
uchumi      kitaifa      133
kitaifa     kimataifa     70
afya        kitaifa       58
kitaifa     michezo       54
michezo     burudani      51
            kitaifa       37
kitaifa     burudani      21
kimataifa   kitaifa       21
dtype: int64

In [14]:
errors.to_csv(RESULTS_DIR / "baseline_errors.csv", index=False)   # annotate this file for your report

## 9. Save the baseline for the web app

The Gradio app can load this file straight away, so you have a working demo before the stronger models are ready. Swap in the best model later.

In [15]:
joblib.dump({"model": baseline, "label_names": label_names, "clean_fn": "clean_text"}, MODELS_DIR / "swahili_baseline.joblib")
print("Saved", MODELS_DIR / "swahili_baseline.joblib")

Saved swahili_baseline.joblib


In [16]:
def pack_metrics(df):
    preds = baseline.predict(df["clean"])
    acc = accuracy_score(df["label"], preds)
    p, r, f1, _ = precision_recall_fscore_support(df["label"], preds, average="macro", zero_division=0)
    return {
        "accuracy": round(float(acc), 4),
        "macro_p": round(float(p), 4),
        "macro_r": round(float(r), 4),
        "macro_f1": round(float(f1), 4),
    }

import json

ckpt = MODELS_DIR / "swahili_baseline.joblib"
baseline_metrics = {
    "model": "TF-IDF + Logistic Regression",
    "val": pack_metrics(val_df),
    "test": pack_metrics(test_df),
    "train_minutes": round(train_seconds / 60, 2),
    "n_params": int(baseline.named_steps["clf"].coef_.size + baseline.named_steps["clf"].intercept_.size),
    "checkpoint_mb": round(ckpt.stat().st_size / 1e6, 2),
}
(RESULTS_DIR / "baseline_metrics.json").write_text(json.dumps(baseline_metrics, indent=2))
print(json.dumps(baseline_metrics, indent=2))


{
  "model": "TF-IDF + Logistic Regression",
  "val": {
    "accuracy": 0.8891,
    "macro_p": 0.8015,
    "macro_r": 0.8611,
    "macro_f1": 0.8272
  },
  "test": {
    "accuracy": 0.8803,
    "macro_p": 0.7876,
    "macro_r": 0.8426,
    "macro_f1": 0.8114
  },
  "train_minutes": 3.96,
  "n_params": 1200006,
  "checkpoint_mb": 17.92
}


## 10. Quick try-out

In [17]:
def predict(text):
    p = baseline.predict_proba([clean_text(text)])[0]
    top = np.argsort(p)[::-1][:3]
    return [(label_names[i], round(float(p[i]), 3)) for i in top]

predict("Timu ya taifa ya mpira wa miguu imeshinda mechi ya jana usiku")

[('michezo', 0.983), ('kitaifa', 0.007), ('burudani', 0.004)]

## Next steps

1. Model 2: fastText Swahili vectors + BiGRU (same splits, same metrics). See `swahili_news_bigru.ipynb`.
2. Model 3: fine-tune AfroXLMR on a GPU runtime. See `swahili_news_afroxlmr.ipynb`.
3. Comparison table: accuracy, macro-F1, training time, model size. See `swahili_news_comparison.ipynb`.
4. Gradio app (`app.py`) loading the best model, deployed on Hugging Face Spaces. See `app.py`.
5. Report: data source, class definitions, preprocessing, splits, models, metrics, error analysis
